In [1]:
"""
Agent 2: Concept-Performer Knowledge Graph Agent (v2 Standalone)
================================================================
Constructs a Bipartite NetworkX Knowledge Graph connecting Institutional Performers
and Authors to Material Knowledge Concepts and Energy Reaction Types. Computes NetworkX
centrality metrics (PageRank, Betweenness Centrality, Degree) and exports quantitative 
leaderboard workbooks and network visual plots.
"""

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import os

class KnowledgeGraphAgent:
    def __init__(self, data_path="agent1_processed_catalysts_3000.xlsx"):
        self.data_path = data_path
        self.df = None
        self.G = nx.Graph()
        self.pagerank_scores = {}
        self.betweenness_scores = {}
        self.degree_scores = {}
        
    def load_data(self):
        """Loads processed dataset from CSV or Excel across candidate paths."""
        candidate_paths = [
            self.data_path,
            self.data_path.replace(".xlsx", ".csv"),
            "/workspace/artifacts/agent1_processed_catalysts_3000.xlsx",
            "/workspace/scratch/agent1_processed_catalysts_3000.xlsx",
            "/workspace/artifacts/agent1_processed_catalysts_3000.csv",
            "/workspace/scratch/agent1_processed_catalysts_3000.csv"
        ]
        
        found_path = None
        for p in candidate_paths:
            if os.path.exists(p):
                found_path = p
                break
                
        if not found_path:
            raise FileNotFoundError(f"Could not locate dataset in any of: {candidate_paths}")
            
        print(f"[Agent 2] Loading dataset from: {found_path}")
        if found_path.endswith(".csv"):
            self.df = pd.read_csv(found_path)
        else:
            self.df = pd.read_excel(found_path)
            
        print(f"[Agent 2] Ingested {len(self.df)} processed records.")

    def build_bipartite_graph(self, top_n_institutions=15):
        """
        Constructs a Bipartite Graph G = (V_P union V_K, E, W) linking 
        Institutional Performers to Knowledge Concepts (Material Classes & Reaction Types).
        """
        self.G = nx.Graph()
        
        # Determine column names dynamically
        cols = self.df.columns
        mat_col = 'extracted_materials' if 'extracted_materials' in cols else ('material_entities' if 'material_entities' in cols else cols[0])
        rxn_col = 'extracted_reactions' if 'extracted_reactions' in cols else ('reaction_type' if 'reaction_type' in cols else cols[1])

        # 1. Filter top institutions
        inst_counts = {}
        for inst_str in self.df['institutions'].dropna():
            for inst in str(inst_str).split(';'):
                inst_clean = inst.strip()
                if inst_clean and len(inst_clean) > 3:
                    inst_counts[inst_clean] = inst_counts.get(inst_clean, 0) + 1
                    
        top_insts = set(sorted(inst_counts, key=inst_counts.get, reverse=True)[:top_n_institutions])
        print(f"[Agent 2] Isolated Top {len(top_insts)} Institutional Research Hubs.")

        # 2. Add edges between Performers and Material Concept Nodes
        for idx, row in self.df.iterrows():
            mat_entity = str(row.get(mat_col, 'Unassigned Material')).strip()
            
            concept_node = f"{mat_entity}"
            if not self.G.has_node(concept_node):
                self.G.add_node(concept_node, node_type='Concept', category='Material')
                
            row_insts = [i.strip() for i in str(row.get('institutions', '')).split(';') if i.strip()]
            for inst in row_insts:
                if inst in top_insts:
                    performer_node = f"{inst}"
                    if not self.G.has_node(performer_node):
                        self.G.add_node(performer_node, node_type='Performer', category='Institution')
                        
                    if self.G.has_edge(performer_node, concept_node):
                        self.G[performer_node][concept_node]['weight'] += 1
                    else:
                        self.G.add_edge(performer_node, concept_node, weight=1)

        print(f"[Agent 2] Knowledge Graph Constructed: {self.G.number_of_nodes()} Nodes, {self.G.number_of_edges()} Edges.")

    def compute_centrality_metrics(self):
        """Computes PageRank Power Iteration, Betweenness Centrality, and Degree Centrality."""
        print("[Agent 2] Computing PageRank Stationary Distribution (AIMA Ch. 3/12)...")
        self.pagerank_scores = nx.pagerank(self.G, weight='weight', alpha=0.85)
        
        print("[Agent 2] Computing Betweenness Centrality...")
        self.betweenness_scores = nx.betweenness_centrality(self.G, weight='weight')
        
        print("[Agent 2] Computing Weighted Degree Centrality...")
        self.degree_scores = dict(self.G.degree(weight='weight'))

    def export_results(self, output_excel="agent2_knowledge_graph_results.xlsx"):
        """Exports graph topology leaderboard metrics to Excel."""
        records = []
        for node in self.G.nodes():
            n_type = self.G.nodes[node].get('node_type', 'Unknown')
            records.append({
                'Node': node,
                'Node_Type': n_type,
                'PageRank_Centrality': self.pagerank_scores.get(node, 0.0),
                'Betweenness_Centrality': self.betweenness_scores.get(node, 0.0),
                'Weighted_Degree': self.degree_scores.get(node, 0)
            })
            
        df_res = pd.DataFrame(records).sort_values(by='PageRank_Centrality', ascending=False)
        df_res.to_excel(output_excel, index=False)
        print(f"[Agent 2] Graph metrics exported to: {output_excel}")
        return df_res

    def generate_network_plot(self, output_img="agent2_network_analysis.png"):
        """Generates publication-quality network graph topology and PageRank leaderboard figure."""
        plt.figure(figsize=(16, 7))
        
        # Subplot 1: Bipartite Network Layout
        ax1 = plt.subplot(1, 2, 1)
        pos = nx.spring_layout(self.G, k=0.6, seed=42)
        
        performer_nodes = [n for n, d in self.G.nodes(data=True) if d.get('node_type') == 'Performer']
        concept_nodes = [n for n, d in self.G.nodes(data=True) if d.get('node_type') == 'Concept']
        
        p_sizes = [self.pagerank_scores.get(n, 0.01) * 25000 for n in performer_nodes]
        c_sizes = [self.pagerank_scores.get(n, 0.01) * 25000 for n in concept_nodes]
        
        nx.draw_networkx_nodes(self.G, pos, nodelist=performer_nodes, node_color='#1f77b4', 
                              node_size=p_sizes, alpha=0.85, label='Institutional Performers', ax=ax1)
        nx.draw_networkx_nodes(self.G, pos, nodelist=concept_nodes, node_color='#ff7f0e', 
                              node_size=c_sizes, alpha=0.85, label='Knowledge Concepts', ax=ax1)
        
        edges = self.G.edges(data=True)
        weights = [d['weight'] for u, v, d in edges]
        max_w = max(weights) if weights else 1
        widths = [1 + 3 * (w / max_w) for w in weights]
        
        nx.draw_networkx_edges(self.G, pos, width=widths, alpha=0.3, edge_color='#7f7f7f', ax=ax1)
        nx.draw_networkx_labels(self.G, pos, font_size=8, font_weight='bold', ax=ax1)
        
        ax1.set_title("Agent 2: Bipartite Concept-Performer Knowledge Graph\n(Node size proportional to PageRank Centrality)", fontsize=11, fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.axis('off')

        # Subplot 2: PageRank Leaderboard Bar Chart
        ax2 = plt.subplot(1, 2, 2)
        df_res = pd.DataFrame([{
            'Node': n, 
            'PageRank': self.pagerank_scores.get(n, 0.0),
            'Type': self.G.nodes[n].get('node_type')
        } for n in self.G.nodes()]).sort_values(by='PageRank', ascending=False).head(12)
        
        sns.barplot(data=df_res, x='PageRank', y='Node', hue='Type', dodge=False, palette='viridis', ax=ax2)
        ax2.set_title("Top 12 Performing Nodes by PageRank Centrality", fontsize=11, fontweight='bold')
        ax2.set_xlabel("PageRank Score")
        ax2.grid(True, linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.savefig(output_img, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"[Agent 2] Network visual plot saved to: {output_img}")

    def run_pipeline(self):
        """Executes full Agent 2 pipeline workflow."""
        self.load_data()
        self.build_bipartite_graph()
        self.compute_centrality_metrics()
        self.export_results()
        self.generate_network_plot()
        print("[Agent 2] Execution complete!")

if __name__ == "__main__":
    agent = KnowledgeGraphAgent()
    agent.run_pipeline()


[Agent 2] Loading dataset from: agent1_processed_catalysts_3000.xlsx
[Agent 2] Ingested 3000 processed records.
[Agent 2] Isolated Top 12 Institutional Research Hubs.
[Agent 2] Knowledge Graph Constructed: 21 Nodes, 108 Edges.
[Agent 2] Computing PageRank Stationary Distribution (AIMA Ch. 3/12)...
[Agent 2] Computing Betweenness Centrality...
[Agent 2] Computing Weighted Degree Centrality...
[Agent 2] Graph metrics exported to: agent2_knowledge_graph_results.xlsx
[Agent 2] Network visual plot saved to: agent2_network_analysis.png
[Agent 2] Execution complete!
